In [18]:
# Data handling
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler


In [19]:
data = pd.read_csv("data.csv", encoding="latin1")


# Show first 5 rows
data.head(11)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
5,536365,22752,SET 7 BABUSHKA NESTING BOXES,2,12/1/2010 8:26,7.65,17850.0,United Kingdom
6,536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,12/1/2010 8:26,4.25,17850.0,United Kingdom
7,536366,22633,HAND WARMER UNION JACK,6,12/1/2010 8:28,1.85,17850.0,United Kingdom
8,536366,22632,HAND WARMER RED POLKA DOT,6,12/1/2010 8:28,1.85,17850.0,United Kingdom
9,536367,84879,ASSORTED COLOUR BIRD ORNAMENT,32,12/1/2010 8:34,1.69,13047.0,United Kingdom


In [20]:
# Remove rows with missing CustomerID
data = data.dropna(subset=['CustomerID'])

# Remove cancelled orders (negative quantity)
data = data[data['Quantity'] > 0]

# Convert InvoiceDate to datetime
data['InvoiceDate'] = pd.to_datetime(data['InvoiceDate'])


In [21]:
# Create TotalPrice column
data['TotalPrice'] = data['Quantity'] * data['UnitPrice']

data.head(11)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalPrice
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
5,536365,22752,SET 7 BABUSHKA NESTING BOXES,2,2010-12-01 08:26:00,7.65,17850.0,United Kingdom,15.30
6,536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,2010-12-01 08:26:00,4.25,17850.0,United Kingdom,25.50
7,536366,22633,HAND WARMER UNION JACK,6,2010-12-01 08:28:00,1.85,17850.0,United Kingdom,11.10
8,536366,22632,HAND WARMER RED POLKA DOT,6,2010-12-01 08:28:00,1.85,17850.0,United Kingdom,11.10
9,536367,84879,ASSORTED COLOUR BIRD ORNAMENT,32,2010-12-01 08:34:00,1.69,13047.0,United Kingdom,54.08


In [22]:
# Create customer-based features
customer_data = data.groupby('CustomerID').agg({
    'InvoiceNo': 'nunique',      # Number of purchases
    'Quantity': 'sum',           # Total items bought
    'TotalPrice': 'sum'          # Total money spent
}).reset_index()

customer_data.head()


,CustomerID,InvoiceNo,Quantity,TotalPrice
0,12346.0,1,74215,77183.60
1,12347.0,7,2458,4310.00
2,12348.0,4,2341,1797.24
3,12349.0,1,631,1757.55
4,12350.0,1,197,334.40


In [23]:
customer_data.columns = [
    'CustomerID',
    'Number_of_Orders',
    'Total_Quantity',
    'Total_Spending'
]

customer_data.head()


,CustomerID,Number_of_Orders,Total_Quantity,Total_Spending
0,12346.0,1,74215,77183.60
1,12347.0,7,2458,4310.00
2,12348.0,4,2341,1797.24
3,12349.0,1,631,1757.55
4,12350.0,1,197,334.40


In [24]:
X = customer_data[['Number_of_Orders', 'Total_Quantity', 'Total_Spending']]


In [25]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


In [26]:
from sklearn.cluster import KMeans

# Train K-Means model
kmeans = KMeans(n_clusters=4, random_state=42)
kmeans.fit(X_scaled)


KMeans(n_clusters=4, random_state=42)

# STEP 2: Test the Machine Learning Model

In [27]:
# Add cluster labels to customer data
customer_data['Cluster'] = kmeans.labels_

customer_data.head()


,CustomerID,Number_of_Orders,Total_Quantity,Total_Spending,Cluster
0,12346.0,1,74215,77183.60,2
1,12347.0,7,2458,4310.00,0
2,12348.0,4,2341,1797.24,0
3,12349.0,1,631,1757.55,0
4,12350.0,1,197,334.40,0


In [28]:
# Count customers in each cluster
customer_data['Cluster'].value_counts()


Cluster
0    3964
3     345
2      24
1       6
Name: count, dtype: int64

## Test_Model

In [29]:
# Example customer data (Number of Orders, Quantity, Total Spending)
sample_customer = [[5, 1500, 3000]]

# Scale the sample using the same scaler
sample_scaled = scaler.transform(sample_customer)

# Predict cluster
predicted_cluster = kmeans.predict(sample_scaled)

predicted_cluster


C:\Users\Shayan_Riaz\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


array([0], dtype=int32)

### If You Want to Remove the Warning

In [32]:
# Create sample customer using SAME features as training
sample_customer = pd.DataFrame(
    [[5, 1500, 3000]],
    columns=['Number_of_Orders', 'Total_Quantity', 'Total_Spending']
)

# Scale using trained scaler
sample_scaled = scaler.transform(sample_customer)

# Predict cluster
predicted_cluster = kmeans.predict(sample_scaled)

predicted_cluster


array([0], dtype=int32)

“While testing the model, I used the same features that were used during training to ensure consistency and correct predictions.”

In [31]:
cluster_meaning = {
    0: "Low-value customers",
    1: "Medium-value customers",
    2: "High-value customers",
    3: "Very high-value customers"
}

cluster_meaning[predicted_cluster[0]]


'Low-value customers'

# STEP_3 Save the Trained Machine Learning Model

In [34]:
import joblib

In [35]:
# Save trained KMeans model
joblib.dump(kmeans, "kmeans_customer_model.pkl")


['kmeans_customer_model.pkl']

In [36]:
# Save the scaler
joblib.dump(scaler, "scaler.pkl")


['scaler.pkl']

In [37]:
# Save feature names used during training
features = ['Number_of_Orders', 'Total_Quantity', 'Total_Spending']
joblib.dump(features, "features.pkl")


['features.pkl']

kmeans_customer_model.pkl
scaler.pkl
features.pkl
